# Module 5: Spark SQL

Spark lets you query DataFrames with **standard SQL**. Under the hood, both the DataFrame API and SQL go through the same **Catalyst optimizer** — they produce identical execution plans.

**When to use SQL vs DataFrame API?**
- SQL is great for ad-hoc exploration and when the query reads naturally as SQL
- DataFrame API is better for programmatic pipelines and complex logic
- In practice, teams mix both — use whatever reads clearest

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum

spark = SparkSession.builder \
    .appName("Module 05 - Spark SQL") \
    .master("local[*]") \
    .getOrCreate()

employees = spark.read.csv("../data/employees.csv", header=True, inferSchema=True)
departments = spark.read.csv("../data/departments.csv", header=True, inferSchema=True)
sales = spark.read.csv("../data/sales.csv", header=True, inferSchema=True)

---
## Concept 1: Registering Temp Views

Before using SQL, you must register DataFrames as **temporary views** — virtual tables that exist for the duration of your SparkSession.

`createOrReplaceTempView("name")` registers a DataFrame as a SQL table you can query.

In [2]:
employees.createOrReplaceTempView("employees")
departments.createOrReplaceTempView("departments")
sales.createOrReplaceTempView("sales")

# Verify: list all registered tables
spark.sql("SHOW TABLES").show()

+---------+-----------+-----------+
|namespace|  tableName|isTemporary|
+---------+-----------+-----------+
|         |departments|       true|
|         |  employees|       true|
|         |      sales|       true|
+---------+-----------+-----------+



#### Try It: Register and Verify

The views are already registered. Run `spark.sql("SHOW TABLES").show()` and confirm you see all 3 tables.

Then run `spark.sql("DESCRIBE employees").show()` to see the schema as SQL sees it.

In [3]:
# Your code here


In [4]:
# --- Solution ---
spark.sql("SHOW TABLES").show()
spark.sql("DESCRIBE employees").show()

+---------+-----------+-----------+
|namespace|  tableName|isTemporary|
+---------+-----------+-----------+
|         |departments|       true|
|         |  employees|       true|
|         |      sales|       true|
+---------+-----------+-----------+

+-------------+---------+-------+
|     col_name|data_type|comment|
+-------------+---------+-------+
|  employee_id|      int|   NULL|
|         name|   string|   NULL|
|department_id|      int|   NULL|
|       salary|      int|   NULL|
|    hire_date|     date|   NULL|
|         city|   string|   NULL|
+-------------+---------+-------+



---
## Concept 2: Basic SELECT / WHERE / GROUP BY

`spark.sql("...")` runs a SQL query and returns a DataFrame. You can chain `.show()`, `.count()`, or any DataFrame method on the result.

In [5]:
# High earners sorted by salary
spark.sql("""
    SELECT name, salary, city
    FROM employees
    WHERE salary > 90000
    ORDER BY salary DESC
""").show()

+-------------+------+-------------+
|         name|salary|         city|
+-------------+------+-------------+
|  Peter Lewis|115000|San Francisco|
|    Grace Lee|110000|San Francisco|
|Carol Johnson|105000|San Francisco|
|Victor Wright|102000|San Francisco|
|Zara Mitchell| 99000|San Francisco|
|  Karen White| 98000|San Francisco|
| Frank Wilson| 95000|San Francisco|
|  Yuki Tanaka| 93000|San Francisco|
|   Alice Chen| 92000|San Francisco|
|  Rachel Hall| 91000|San Francisco|
+-------------+------+-------------+



In [6]:
# Aggregations with GROUP BY
spark.sql("""
    SELECT city,
           COUNT(*) AS num_employees,
           ROUND(AVG(salary), 2) AS avg_salary,
           MAX(salary) AS max_salary
    FROM employees
    GROUP BY city
    ORDER BY avg_salary DESC
""").show()

+-------------+-------------+----------+----------+
|         city|num_employees|avg_salary|max_salary|
+-------------+-------------+----------+----------+
|San Francisco|           11|  98909.09|    115000|
|     New York|            9|  76222.22|     85000|
|      Chicago|           10|   68100.0|     76000|
+-------------+-------------+----------+----------+



#### Try It: Write SQL Queries

1. Find all employees in **San Francisco** earning **more than $90,000**. Show name, salary.
2. Count the number of sales per `region`, ordered by count descending.

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
spark.sql("""
    SELECT name, salary
    FROM employees
    WHERE city = 'San Francisco' AND salary > 90000
    ORDER BY salary DESC
""").show()

spark.sql("""
    SELECT region, COUNT(*) AS num_sales
    FROM sales
    GROUP BY region
    ORDER BY num_sales DESC
""").show()

---
## Concept 3: Joins in SQL

SQL joins work exactly like DataFrame joins — same types (INNER, LEFT, RIGHT, FULL OUTER), same performance implications.

In [ ]:
# Two-table join
spark.sql("""
    SELECT e.name, d.department_name, e.salary, d.location
    FROM employees e
    JOIN departments d ON e.department_id = d.department_id
    ORDER BY e.salary DESC
""").show(10)

In [ ]:
# Three-table join: revenue by department
spark.sql("""
    SELECT d.department_name,
           ROUND(SUM(s.amount), 2) AS total_revenue,
           COUNT(*) AS num_sales
    FROM sales s
    JOIN employees e ON s.employee_id = e.employee_id
    JOIN departments d ON e.department_id = d.department_id
    GROUP BY d.department_name
    ORDER BY total_revenue DESC
""").show()

#### Try It: SQL Joins

Write a SQL query that joins `employees` to `departments` and shows: `name`, `department_name`, `location`, `salary`. Filter for `location = 'New York'` only.

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
spark.sql("""
    SELECT e.name, d.department_name, d.location, e.salary
    FROM employees e
    JOIN departments d ON e.department_id = d.department_id
    WHERE d.location = 'New York'
    ORDER BY e.salary DESC
""").show()

---
## Concept 4: CTEs (Common Table Expressions)

CTEs (`WITH` clauses) let you break complex queries into readable, named subqueries. Essential for production SQL — they make complex logic maintainable.

In [ ]:
# CTE: Find departments where average salary exceeds the company average
spark.sql("""
    WITH dept_avg AS (
        SELECT e.department_id,
               d.department_name,
               ROUND(AVG(e.salary), 2) AS dept_avg_salary
        FROM employees e
        JOIN departments d ON e.department_id = d.department_id
        GROUP BY e.department_id, d.department_name
    ),
    company_avg AS (
        SELECT ROUND(AVG(salary), 2) AS overall_avg
        FROM employees
    )
    SELECT da.department_name,
           da.dept_avg_salary,
           ca.overall_avg
    FROM dept_avg da
    CROSS JOIN company_avg ca
    WHERE da.dept_avg_salary > ca.overall_avg
    ORDER BY da.dept_avg_salary DESC
""").show()

#### Try It: Write a CTE

Write a CTE that:
1. First CTE (`monthly_sales`): compute total revenue per month (use `SUBSTRING(sale_date, 1, 7)` for year-month)
2. Main query: select from `monthly_sales`, order by month

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
spark.sql("""
    WITH monthly_sales AS (
        SELECT SUBSTRING(sale_date, 1, 7) AS sale_month,
               ROUND(SUM(amount), 2) AS total_revenue,
               COUNT(*) AS num_sales
        FROM sales
        GROUP BY SUBSTRING(sale_date, 1, 7)
    )
    SELECT * FROM monthly_sales
    ORDER BY sale_month
""").show()

---
## Concept 5: SQL vs DataFrame API — Same Execution Plan

Use `.explain()` to see the physical execution plan. Both approaches produce the same plan because Catalyst optimizes both identically. This means: **use whichever reads better — there's no performance difference**.

In [7]:
# DataFrame API version
df_result = (
    sales.join(employees, "employee_id")
    .groupBy("city")
    .agg(sum("amount").alias("total"))
)
print("=== DataFrame API Plan ===")
df_result.explain()

=== DataFrame API Plan ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[city#22], functions=[sum(amount#64)])
   +- Exchange hashpartitioning(city#22, 200), ENSURE_REQUIREMENTS, [plan_id=192]
      +- HashAggregate(keys=[city#22], functions=[partial_sum(amount#64)])
         +- Project [amount#64, city#22]
            +- BroadcastHashJoin [employee_id#62], [employee_id#17], Inner, BuildRight, false
               :- Filter isnotnull(employee_id#62)
               :  +- FileScan csv [employee_id#62,amount#64] Batched: false, DataFilters: [isnotnull(employee_id#62)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/data/sales.csv], PartitionFilters: [], PushedFilters: [IsNotNull(employee_id)], ReadSchema: struct<employee_id:int,amount:double>
               +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [plan_id=187]
                  +- Filter isnotnull(employee_id#17)
             

In [8]:
# SQL version of the SAME query
sql_result = spark.sql("""
    SELECT e.city, SUM(s.amount) AS total
    FROM sales s
    JOIN employees e ON s.employee_id = e.employee_id
    GROUP BY e.city
""")
print("=== SQL Plan ===")
sql_result.explain()

=== SQL Plan ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[city#22], functions=[sum(amount#64)])
   +- Exchange hashpartitioning(city#22, 200), ENSURE_REQUIREMENTS, [plan_id=234]
      +- HashAggregate(keys=[city#22], functions=[partial_sum(amount#64)])
         +- Project [amount#64, city#22]
            +- BroadcastHashJoin [employee_id#62], [employee_id#17], Inner, BuildRight, false
               :- Filter isnotnull(employee_id#62)
               :  +- FileScan csv [employee_id#62,amount#64] Batched: false, DataFilters: [isnotnull(employee_id#62)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/data/sales.csv], PartitionFilters: [], PushedFilters: [IsNotNull(employee_id)], ReadSchema: struct<employee_id:int,amount:double>
               +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [plan_id=229]
                  +- Filter isnotnull(employee_id#17)
                     +-

#### Try It: Compare Plans

Write the same query two ways — find the total revenue per product:
1. Using the DataFrame API: `sales.groupBy("product").agg(sum("amount"))`
2. Using SQL: `SELECT product, SUM(amount) FROM sales GROUP BY product`

Call `.explain()` on both and compare.

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
df_way = sales.groupBy("product").agg(sum("amount").alias("total"))
print("DataFrame API:")
df_way.explain()

sql_way = spark.sql("SELECT product, SUM(amount) AS total FROM sales GROUP BY product")
print("\nSQL:")
sql_way.explain()

---
## Capstone Exercise

Write a SQL query using a **window function** to rank employees by salary within each department.

Use `RANK() OVER (PARTITION BY department_name ORDER BY salary DESC)` to assign ranks.

Show: department_name, name, salary, rank. Order by department then rank.

In [ ]:
# Your capstone code here


In [ ]:
# --- Capstone Solution ---
spark.sql("""
    SELECT d.department_name,
           e.name,
           e.salary,
           RANK() OVER (PARTITION BY d.department_name ORDER BY e.salary DESC) AS salary_rank
    FROM employees e
    JOIN departments d ON e.department_id = d.department_id
    ORDER BY d.department_name, salary_rank
""").show(30)

In [ ]:
spark.stop()

---
## What You Learned

- **`createOrReplaceTempView`** registers DataFrames as SQL tables
- **`spark.sql("...")`** runs SQL and returns a DataFrame
- SQL supports **SELECT, WHERE, GROUP BY, JOIN, ORDER BY** — all standard SQL
- **CTEs** (`WITH` clauses) break complex queries into readable parts
- **Window functions** (`RANK`, `ROW_NUMBER`, etc.) rank within groups
- SQL and DataFrame API produce **identical execution plans** — no performance difference

Next: **Module 6 — Reading and Writing Data**, where you'll work with Parquet, JSON, and partitioned data.